<a href="https://colab.research.google.com/github/sudo-ansh007/forecast/blob/main/win_probability_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deal Win Probability — forecast_v0

Trains on closed deals, outputs a calibrated win probability per open deal.

**Upload two files:** `ml_train.csv` (11,051 closed deals, labelled) and
`ml_score.csv` (1,912 open deals, unlabelled). Nothing else needed —
this notebook is self-contained.

---

### ⚠️ Before you upload

These CSVs hold **real customer CRM data** — deal values, deal IDs, segments.
Colab is Google-hosted and notebooks are shareable by link. Clear it with
whoever owns data governance, or run this locally in Jupyter instead
(identical code, no third-party exposure).

Never paste an API token into a cell — notebooks save their output.

## 1. Setup

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")   # sigmoid calibrator overflows harmlessly at 4.9% positives

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
IN_COLAB = "google.colab" in sys.modules
print(f"colab={IN_COLAB}  pandas={pd.__version__}")

In [ ]:
import os

if IN_COLAB and not os.path.exists("ml_train.csv"):
    from google.colab import files
    print("Upload ml_train.csv and ml_score.csv")
    files.upload()

# Sort by created_date NOW so every split below is a time split. CSV row order is
# whatever the source table returned -- a positional split on it is quasi-random and
# inflates PR-AUC from 0.56 to 0.75 by putting a deal's future neighbours in train.
train = pd.read_csv("ml_train.csv").sort_values("created_date").reset_index(drop=True)
score = pd.read_csv("ml_score.csv")

FEATURES = ["days_in_current_stage", "days_in_sales_cycle", "stall_ratio",
            "contract_months", "num_stakeholders", "has_champion", "poc",
            "opportunity_type", "source", "geo", "segment"]
CATEGORICALS = ["opportunity_type", "source", "geo", "segment"]
LABEL = "is_won"

# display_id, acv and created_date ride along for joining / revenue weighting /
# splitting -- NOT features. acv is filled after signing (93% of won deals vs 29%
# of lost) and created_date carries cohort censoring, so either would leak.
assert set(FEATURES).issubset(train.columns), set(FEATURES) - set(train.columns)

print(f"train  {len(train):>6,} closed deals   {train[LABEL].sum()} won "
      f"({train[LABEL].mean():.1%} win rate)")
print(f"score  {len(score):>6,} open deals     no label")
train.head()

## 2. What the features mean

| Feature | Means |
|---|---|
| `days_in_current_stage` | Days since the stage last changed |
| `days_in_sales_cycle` | Days since the deal was created |
| `stall_ratio` | `days_in_current_stage / days_in_sales_cycle` — share of the deal's life parked in one stage. **Derived, and the top driver** |
| `contract_months` | Contract term |
| `num_stakeholders` | Contacts on the deal |
| `has_champion` | A champion is named |
| `poc` | POC recorded (checkbox — see §7) |
| `opportunity_type` | new / renewal / upsell / amendment |
| `source` | outbound / inbound / events / partner / … |
| `geo` | amer / apj / emea |
| `segment` | enterprise / mid-market / smb / start-up / 2k |

Deliberately excluded: `acv` (filled after signing), `stage` (on a closed deal the
stage *is* the label), raw `created_date` (cohort censoring), `target_close_date`
(overwritten to the actual close date on 99.3% of closed deals), and 10 fields that
are only populated *because* a deal closed.

In [ ]:
print("win rate by feature value:\n")
for c in ["has_champion", "poc"]:
    g = train.groupby(c)[LABEL].agg(deals="size", wins="sum", win_rate="mean")
    lift = g.loc[1, "win_rate"] / g.loc[0, "win_rate"]
    print(f"{c}  ({lift:.1f}x lift)")
    print(g.assign(win_rate=g.win_rate.map("{:.2%}".format)).to_string(), "\n")

# opportunity_type separates harder than anything else here -- and that is the problem.
# Renewals and new business are two different sales processes; one model with one
# coefficient averages them, and 80%+ of the open pipeline is the low-rate population.
for c in ["opportunity_type", "segment", "source", "geo"]:
    g = train.groupby(c)[LABEL].agg(deals="size", wins="sum", win_rate="mean")
    g["pct_of_open"] = score[c].value_counts(normalize=True).reindex(g.index).fillna(0)
    g = g.sort_values("win_rate", ascending=False)
    spread = g.win_rate.max() / max(g.win_rate.min(), 1e-9)
    print(f"{c}  ({spread:.1f}x spread across levels)")
    print(g.assign(win_rate=g.win_rate.map("{:.2%}".format),
                   pct_of_open=g.pct_of_open.map("{:.1%}".format)).to_string(), "\n")

print("stall_ratio  (share of life parked in one stage)")
print(train.groupby(LABEL)["stall_ratio"].median().rename("median").to_string())
print("  0 = lost, 1 = won. Winners advance; losers park.")

## 3. Encode

One-hot the four categoricals. The `reindex` matters: a category that appears only
in the scoring set would otherwise shift column order and silently mis-align every
feature. Using native categorical support (LightGBM/CatBoost) instead is fine —
just keep train and score consistent.

In [ ]:
def encode(df_train, df_other):
    num = [f for f in FEATURES if f not in CATEGORICALS]

    def build(df):
        out = df[num].astype(float).copy()
        for c in CATEGORICALS:
            out = pd.concat([out, pd.get_dummies(df[c], prefix=c, dtype=float)], axis=1)
        return out

    a, b = build(df_train), build(df_other)

    # The reindex below silently zeroes any level the model never trained on, so such a
    # deal scores as if the field were blank. Surface it. Also flag levels the model has
    # <=1 positive for but that carry real open-pipeline volume -- predictions there are
    # unsupported no matter what the calibration curve says.
    for c in CATEGORICALS:
        for lvl in sorted(set(df_other[c].unique()) - set(df_train[c].unique())):
            print(f"  WARNING unseen level {c}={lvl!r} on {(df_other[c]==lvl).sum()} "
                  "open deals -- scored as all-zeros, no training support")
        share = df_other[c].value_counts(normalize=True)
        w = df_train.groupby(c)[LABEL].agg(["size", "sum"])
        for lvl, r in w.iterrows():
            if r["sum"] <= 1 and share.get(lvl, 0) > 0.02:
                print(f"  WARNING thin level {c}={lvl!r}: {int(r['sum'])} win(s) in "
                      f"{int(r['size'])} closed, but {share[lvl]:.0%} of open pipeline")

    b = b.reindex(columns=a.columns, fill_value=0.0)   # align, never reorder
    return a, b

X, X_score = encode(train, score)
y = train[LABEL].to_numpy()
print(f"\n{X.shape[1]} encoded columns from {len(FEATURES)} features")

## 4. Honest evaluation first

Split by deal age — oldest 80% train, newest 20% test. **Never random.** A random
split puts a deal's future neighbours in the training set and inflates every metric.

The rows were sorted by `created_date` in §1, so the positional split below is a
time split. Do **not** drop that sort — the raw CSV order is not chronological.

In [ ]:
cut = int(len(train) * 0.8)
Xtr, Xte = X.iloc[:cut], X.iloc[cut:]
ytr, yte = y[:cut], y[cut:]
print(f"train {len(Xtr):,} / {ytr.sum()} won      test {len(Xte):,} / {yte.sum()} won")
if yte.sum() < 60:
    print(f"WARNING: {yte.sum()} test positives -- read differences of <0.02 PR-AUC as noise")

def new_model():
    # Shallow on purpose: ~490 training positives. Deeper overfits.
    # No class_weight -- it wrecks calibration, which matters more than ranking here.
    # Platt (sigmoid) not isotonic: isotonic needs several hundred positives to behave.
    #
    # shuffle=True is load-bearing. A bare cv=3 uses UNSHUFFLED StratifiedKFold, so on
    # chronologically-sorted rows each calibration fold is a different era of the
    # business and the fitted sigmoid depends on row order -- expected wins swung 71.8 /
    # 79.2 / 72.9 across three orderings of identical data, a 10% move in the headline
    # revenue number from sort order alone. Shuffling plus the pre-shuffle in §5 cuts
    # that spread to 0.3 wins.
    return CalibratedClassifierCV(
        HistGradientBoostingClassifier(
            max_depth=3, min_samples_leaf=50, max_iter=200, learning_rate=0.05,
            early_stopping=True, validation_fraction=0.15, random_state=0),
        method="sigmoid", cv=StratifiedKFold(3, shuffle=True, random_state=0))

p_test = new_model().fit(Xtr, ytr).predict_proba(Xte)[:, 1]

def recall_at_top_k(y_true, p, frac=0.2):
    k = max(1, int(len(p) * frac))
    return y_true[np.argsort(p)[::-1][:k]].sum() / y_true.sum()

print(f"""
PR-AUC        {average_precision_score(yte, p_test):.4f}   (random = {yte.mean():.4f})
Brier         {brier_score_loss(yte, p_test):.4f}   calibration -- the metric that matters
ROC-AUC       {roc_auc_score(yte, p_test):.4f}   report only, do not optimise
Recall@20%    {recall_at_top_k(yte, p_test):.4f}   share of real wins in the top-scoring fifth
sum(p)/actual {p_test.sum() / yte.sum():.2f}x    aggregate sanity, want 0.9-1.1""")
print("\nAccuracy is omitted on purpose: at a 4.9% win rate, 'everything loses'")
print("scores 95.1% and finds nothing.")

In [ ]:
# Calibration: does "30%" actually close 30% of the time?
edges = np.linspace(0, 1, 11)
idx = np.clip(np.digitize(p_test, edges) - 1, 0, 9)
rel = pd.DataFrame([
    {"bin": f"{edges[b]:.0%}-{edges[b+1]:.0%}", "deals": int((idx == b).sum()),
     "predicted": p_test[idx == b].mean(), "actual": yte[idx == b].mean()}
    for b in range(10) if (idx == b).sum()
])
display(rel.round(4))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax[0].plot(rel.predicted, rel.actual, "o-", label="model")
for _, r in rel.iterrows():
    ax[0].annotate(f"n={r.deals:.0f}", (r.predicted, r.actual), fontsize=7,
                   xytext=(4, -9), textcoords="offset points")
ax[0].set(xlabel="predicted", ylabel="actual win rate", title="Reliability")
ax[0].legend()

imp = permutation_importance(new_model().fit(Xtr, ytr), Xte, yte,
                             scoring="average_precision", n_repeats=10, random_state=0)
top = pd.Series(imp.importances_mean, index=Xte.columns).nlargest(10).sort_values()
ax[1].barh(top.index, top.values)
ax[1].set(title="Permutation importance (PR-AUC drop when shuffled)")
plt.tight_layout(); plt.show()

print("Thin middle bins (4-24 deals) -- that wobble is small-sample noise, not")
print("miscalibration. The 0-10% bin holds ~2,000 deals and is trustworthy.")

## 4c. Why this model — 20 configurations compared

Every row is a tree model except the logistic regression, which is in as a sanity
floor. Four families: one tree, bagging (random forest / extra trees / bagged tree),
boosting (AdaBoost, HistGB, LightGBM, XGBoost), plus hyperparameter variants of the
shipping config. Each runs 5 seeds; read any gap smaller than the `pr_sd` spread as
noise.

The cell installs `lightgbm` and `xgboost` itself on Colab. Running locally,
`pip install lightgbm xgboost` — on a Mac they also need the OpenMP runtime
(`brew install libomp`). Without them the cell drops 5 rows and still runs.

Expect ~1–2 minutes: 20 configs × 5 seeds, and the calibrator fits each model 3 more
times for its CV folds.

Three things this table is here to settle:
- **Ensembling isn't decoration.** A single tree is the *worst* tree in the set. At 49
  test wins one tree either underfits or memorises a handful of deals.
- **Calibration is not free accuracy.** `scale_pos_weight` and deeper trees can win on
  PR-AUC while making `sum(p)/actual` worse — and that column is the revenue number.
  Rank on `calib_err` and Brier, not PR-AUC alone.
- **A gap inside the seed spread is not a result.** `pr_sd` is printed per row for
  exactly this reason.


In [ ]:
import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, BaggingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if IN_COLAB:
    !pip install -q lightgbm xgboost
try:
    from lightgbm import LGBMClassifier
    from xgboost import XGBClassifier
    HAVE_BOOSTERS = True
except ImportError:
    HAVE_BOOSTERS = False
    print("lightgbm/xgboost not installed -- skipping 5 rows.")
    print("  pip install lightgbm xgboost   (on a Mac also: brew install libomp)")

SEEDS = range(5)
skf = lambda: StratifiedKFold(3, shuffle=True, random_state=0)
cal = lambda m: CalibratedClassifierCV(m, method="sigmoid", cv=skf())
pos_weight = (ytr == 0).sum() / (ytr == 1).sum()   # ~17:1 at a 4.9% win rate

def hgb(s, **kw):
    p = dict(max_depth=3, min_samples_leaf=50, max_iter=200, learning_rate=0.05,
             early_stopping=True, validation_fraction=0.15, random_state=s)
    p.update(kw)
    return HistGradientBoostingClassifier(**p)

FAMILIES = {
    # --- not a tree: the floor any tree has to beat --------------------------
    "logreg L2 (not a tree)":  lambda s: cal(make_pipeline(StandardScaler(),
                                   LogisticRegression(max_iter=2000, C=0.1))),
    # --- one tree, no ensemble ----------------------------------------------
    "single tree depth3":      lambda s: cal(DecisionTreeClassifier(max_depth=3,
                                   min_samples_leaf=50, random_state=s)),
    "single tree unlimited":   lambda s: cal(DecisionTreeClassifier(random_state=s)),
    # --- bagging: average many independent trees -----------------------------
    "random forest 500":       lambda s: cal(RandomForestClassifier(n_estimators=500,
                                   min_samples_leaf=50, random_state=s, n_jobs=-1)),
    "random forest balanced":  lambda s: cal(RandomForestClassifier(n_estimators=500,
                                   min_samples_leaf=50, class_weight="balanced",
                                   random_state=s, n_jobs=-1)),
    "extra trees 500":         lambda s: cal(ExtraTreesClassifier(n_estimators=500,
                                   min_samples_leaf=50, random_state=s, n_jobs=-1)),
    "bagged tree depth3":      lambda s: cal(BaggingClassifier(
                                   DecisionTreeClassifier(max_depth=3, min_samples_leaf=50),
                                   n_estimators=200, random_state=s, n_jobs=-1)),
    # --- boosting: each tree fixes the previous ones' mistakes ---------------
    "adaboost 200":            lambda s: cal(AdaBoostClassifier(n_estimators=200,
                                   learning_rate=0.05, random_state=s)),
    "HGB raw (uncalibrated)":  lambda s: hgb(s),
    "HGB+sigmoid  SHIPPING":   lambda s: cal(hgb(s)),
    "HGB+isotonic":            lambda s: CalibratedClassifierCV(hgb(s),
                                   method="isotonic", cv=skf()),
    "HGB depth2 shallower":    lambda s: cal(hgb(s, max_depth=2)),
    "HGB depth8 leaf5":        lambda s: cal(hgb(s, max_depth=8, min_samples_leaf=5)),
    "HGB lr0.02 iter800":      lambda s: cal(hgb(s, learning_rate=0.02, max_iter=800)),
    "HGB no early stopping":   lambda s: cal(hgb(s, early_stopping=False)),
}
if HAVE_BOOSTERS:
    FAMILIES.update({
        "LightGBM":               lambda s: cal(LGBMClassifier(max_depth=3, num_leaves=8,
                                      min_child_samples=50, n_estimators=200, learning_rate=0.05,
                                      random_state=s, verbose=-1, n_jobs=-1)),
        "LightGBM scale_pos_wt":  lambda s: cal(LGBMClassifier(max_depth=3, num_leaves=8,
                                      min_child_samples=50, n_estimators=200, learning_rate=0.05,
                                      scale_pos_weight=pos_weight,
                                      random_state=s, verbose=-1, n_jobs=-1)),
        "LightGBM subsample+reg": lambda s: cal(LGBMClassifier(max_depth=3, num_leaves=8,
                                      min_child_samples=50, n_estimators=200, learning_rate=0.05,
                                      reg_alpha=1.0, reg_lambda=1.0, subsample=0.8,
                                      subsample_freq=1, colsample_bytree=0.8,
                                      random_state=s, verbose=-1, n_jobs=-1)),
        "XGBoost":                lambda s: cal(XGBClassifier(max_depth=3, min_child_weight=50,
                                      n_estimators=200, learning_rate=0.05, random_state=s,
                                      eval_metric="logloss", n_jobs=-1)),
        "XGBoost min_child_wt1":  lambda s: cal(XGBClassifier(max_depth=3, min_child_weight=1,
                                      n_estimators=200, learning_rate=0.05, random_state=s,
                                      eval_metric="logloss", n_jobs=-1)),
    })

rows = []
PREDS = {}   # model -> seed-0 test predictions, kept for the plots in 4d
for name, make in FAMILIES.items():
    pr, br, rc, ratio, secs = [], [], [], [], []
    for s in SEEDS:
        t0 = time.time()
        p = make(s).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        secs.append(time.time() - t0)
        if s == 0:
            PREDS[name] = p
        pr.append(average_precision_score(yte, p))
        br.append(brier_score_loss(yte, p))
        rc.append(roc_auc_score(yte, p))
        ratio.append(p.sum() / yte.sum())
    rows.append({"model": name, "PR_AUC": np.mean(pr), "pr_sd": np.std(pr),
                 "Brier": np.mean(br), "ROC": np.mean(rc),
                 "sum_p_over_actual": np.mean(ratio), "fit_s": np.mean(secs)})
    print(f"  done {name}")

cmp = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
# calib_err is the product metric: how far the revenue roll-up would be off.
cmp["calib_err"] = (cmp.sum_p_over_actual - 1).abs()
print(f"\ntrain {len(Xtr):,}/{ytr.sum()} wins   test {len(Xte):,}/{yte.sum()} wins   "
      f"{len(list(SEEDS))} seeds each   random PR-AUC = {yte.mean():.4f}\n")
display(cmp.round({"PR_AUC": 4, "pr_sd": 4, "Brier": 5, "ROC": 4,
                   "sum_p_over_actual": 2, "fit_s": 2, "calib_err": 2}))

ship = cmp.loc[cmp.model == "HGB+sigmoid  SHIPPING"].iloc[0]
best_pr, best_cal = cmp.iloc[0], cmp.sort_values("calib_err").iloc[0]
print(f"best PR-AUC      {best_pr.model:24}{best_pr.PR_AUC:.4f}  "
      f"(shipping {ship.PR_AUC:.4f}, gap {best_pr.PR_AUC - ship.PR_AUC:+.4f})")
print(f"best calibrated  {best_cal.model:24}sum(p)/actual {best_cal.sum_p_over_actual:.2f}x")
print(f"\nSeed spread on the shipping model is +-{ship.pr_sd:.4f}. Treat any PR-AUC gap")
print("smaller than that as noise -- it is not a reason to switch models.")


## 4d. The same 20 models, as pictures

The table ranks. These plots show *why* a model ranks where it does — and they
surface failures a single number hides.

Six panels:

1. **PR curve** — precision against recall. The only curve that tells you what a
   rep experiences: "of the deals this flagged, how many closed?" Dashed line is
   the 2.2% base rate.
2. **Reliability** — predicted vs actual, decile bins. On the diagonal = "40%
   means 40%". Below it = over-promising. This is the panel that catches a model
   the PR-AUC column calls excellent.
3. **PR-AUC vs calibration error** — the tradeoff, as a scatter. Bottom-right is
   what you want: ranks well *and* the revenue roll-up is right. A model can be
   far right and far up, which means it sorts deals correctly while getting the
   forecast total wrong.
4. **Seed spread** — mean PR-AUC with the ±1 s.d. bar across 5 seeds. Bars that
   overlap are the same model as far as this test set can tell.
5. **Score distributions** — where each model puts its probability mass. A model
   that piles everything at 0.02 has good Brier and is useless.
6. **Cumulative wins captured** — walk the ranked list; how fast do you collect
   the 49 real wins? Diagonal is random. This is the lift a rep actually gets from
   working the list top-down.

Panels 1, 2, 5 and 6 use **seed 0 only**, since a curve can't be averaged
honestly. Panels 3 and 4 use all 5 seeds. Run 4c first — this cell reads `PREDS`
and `cmp` from it.


In [ ]:
if "PREDS" not in globals():
    raise NameError("run section 4c first -- this cell reads PREDS and cmp from it")

order = cmp.model.tolist()                      # best PR-AUC first
colors = plt.cm.viridis(np.linspace(0, 0.92, len(order)))
CLR = dict(zip(order, colors))
SHIP = "HGB+sigmoid  SHIPPING"
lw = lambda m: 2.6 if m == SHIP else 1.2        # shipping model drawn thicker
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(3, 2, figsize=(15, 15))

# 1. precision-recall -- what a rep working the flagged list actually gets
for m in order:
    pre, rec, _ = precision_recall_curve(yte, PREDS[m])
    ax[0, 0].plot(rec, pre, color=CLR[m], lw=lw(m), alpha=0.9)
ax[0, 0].axhline(yte.mean(), ls="--", c="k", lw=1)
ax[0, 0].set(xlabel="recall (share of real wins found)", ylabel="precision",
             title="1. Precision-Recall (seed 0)", ylim=(0, 1))
ax[0, 0].annotate(f"base rate {yte.mean():.1%}", (0.55, yte.mean()),
                  xytext=(0, 6), textcoords="offset points", fontsize=8)

# 2. reliability -- does "40%" close 40% of the time?
edges = np.linspace(0, 1, 11)
for m in order:
    p = PREDS[m]
    b = np.clip(np.digitize(p, edges) - 1, 0, 9)
    pts = [(p[b == k].mean(), yte[b == k].mean()) for k in range(10)
           if (b == k).sum() >= 20]     # <20 deals in a bin is noise, not a point
    if pts:
        ax[0, 1].plot(*zip(*pts), "o-", color=CLR[m], lw=lw(m), ms=4, alpha=0.9)
ax[0, 1].plot([0, 1], [0, 1], "k--", lw=1)
ax[0, 1].set(xlabel="predicted", ylabel="actual win rate",
             title="2. Reliability -- on the diagonal or over-promising (bins >=20 deals)")

# 3. the real tradeoff: ranking quality vs revenue accuracy
for m in order:
    r = cmp.loc[cmp.model == m].iloc[0]
    ax[1, 0].scatter(r.PR_AUC, r.calib_err, color=CLR[m],
                     s=190 if m == SHIP else 70,
                     marker="*" if m == SHIP else "o", zorder=3)
    ax[1, 0].annotate(m, (r.PR_AUC, r.calib_err), fontsize=6.5,
                      xytext=(4, 3), textcoords="offset points")
ax[1, 0].axhline(0, c="k", lw=1)
ax[1, 0].set(xlabel="PR-AUC (higher better) ->", ylabel="|sum(p)/actual - 1|  (lower better)",
             title="3. Ranking vs revenue accuracy -- want bottom-right (* = shipping)")

# 4. seed spread -- overlapping bars mean the gap is not real
c4 = cmp.sort_values("PR_AUC")
ax[1, 1].barh(c4.model, c4.PR_AUC, xerr=c4.pr_sd,
              color=[CLR[m] for m in c4.model], error_kw=dict(lw=1.1, ecolor="k"))
ax[1, 1].axvline(yte.mean(), ls="--", c="r", lw=1)
ax[1, 1].set(xlabel="PR-AUC (bar = mean of 5 seeds, whisker = +-1 s.d.)",
             title="4. Overlapping whiskers = same model, as far as 49 wins can tell")
ax[1, 1].tick_params(labelsize=7)

# 5. where each model puts its probability mass
for m in order:
    ax[2, 0].hist(PREDS[m], bins=np.linspace(0, 1, 41), histtype="step",
                  color=CLR[m], lw=lw(m), alpha=0.9)
ax[2, 0].set(xlabel="predicted win probability", ylabel="test deals (log)",
             yscale="log", title="5. Score distributions -- all mass at zero = useless")

# 6. cumulative wins captured walking the ranked list top-down
n = len(yte)
for m in order:
    hits = np.cumsum(yte[np.argsort(PREDS[m])[::-1]]) / yte.sum()
    ax[2, 1].plot(np.arange(1, n + 1) / n, hits, color=CLR[m], lw=lw(m), alpha=0.9)
ax[2, 1].plot([0, 1], [0, 1], "k--", lw=1)
ax[2, 1].axvline(0.2, ls=":", c="r", lw=1)
ax[2, 1].set(xlabel="share of pipeline worked, best-scored first",
             ylabel=f"share of the {int(yte.sum())} real wins captured",
             title="6. Lift -- dashed is random, red line is the top 20%")

h = [plt.Line2D([], [], color=CLR[m], lw=lw(m)) for m in order]
fig.legend(h, order, loc="lower center", ncol=4, fontsize=7.5,
           frameon=False, bbox_to_anchor=(0.5, -0.035))
plt.tight_layout(); plt.show()

top = cmp.nsmallest(1, "calib_err").iloc[0]
print(f"Panel 3 read-off: best-calibrated is {top.model} at {top.calib_err:.2f} off, "
      f"PR-AUC {top.PR_AUC:.4f}")
print(f"                  best-ranking is {cmp.iloc[0].model} at PR-AUC "
      f"{cmp.iloc[0].PR_AUC:.4f}, but {cmp.iloc[0].calib_err:.2f} off on revenue")
print("\nPanel 2 is the veto. A model that ranks well but sits below the diagonal")
print("hands sales numbers that do not hold up, which is worse than a weaker model")
print("that is honest about its own uncertainty.")


## 5. Train on everything, score the open pipeline

Evaluation is done. Refit on all 11,051 closed deals — more labels, better model —
and score the open ones.

In [ ]:
# Shuffle before the final fit. HistGB's early-stopping holdout is the LAST 15% of rows
# as given, so on the created_date-sorted frame it validates on one era only. Fixed seed
# keeps the run reproducible.
shuf = train.sample(frac=1, random_state=0).reset_index(drop=True)
Xs, X_score = encode(shuf, score)
final = new_model().fit(Xs, shuf[LABEL].to_numpy())

out = score[["display_id", "acv"] + FEATURES].copy()
out["win_probability"] = final.predict_proba(X_score)[:, 1]
out["health"] = pd.cut(out.win_probability, [-0.01, 0.10, 0.30, 1.0],
                       labels=["Red", "Yellow", "Green"])
out["expected_revenue"] = out.win_probability * out.acv
out = out.sort_values("win_probability", ascending=False)

assert out.win_probability.between(0, 1).all()
assert out.health.notna().all()

print(f"scored {len(out):,} open deals")
print(f"  win_probability   min {out.win_probability.min():.3f}  "
      f"median {out.win_probability.median():.3f}  max {out.win_probability.max():.3f}")
print(f"  expected wins     {out.win_probability.sum():.0f}")
print(f"  expected revenue  ${out.expected_revenue.sum()/1e6:.1f}M")
print("    ^ amount is real on ~41% of these deals; the rest is imputed. Range, not point.")
print("\nhealth bands (thresholds are PLACEHOLDERS -- sales owns these numbers):")
print(out.health.value_counts().reindex(["Green", "Yellow", "Red"]).to_string())

out.to_csv("ml_predictions.csv", index=False)
print("\nwrote ml_predictions.csv")
out[["display_id", "win_probability", "health", "acv", "stall_ratio",
     "has_champion", "segment"]].head(10).round(3)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download("ml_predictions.csv")

## 6. The output a sales manager would act on

In [ ]:
risk = out[(out.acv > 50_000) & (out.win_probability < 0.10)] \
         .nlargest(15, "days_in_current_stage")
print(f"AT RISK -- >$50k, <10% win probability, longest stalled")
print(f"${risk.acv.sum()/1e6:.1f}M across these {len(risk)} deals, "
      "all booked at full stage-constant value by today's forecast\n")
display(risk[["display_id", "acv", "win_probability", "days_in_current_stage",
              "stall_ratio", "num_stakeholders", "has_champion", "segment",
              "opportunity_type"]].round(3).reset_index(drop=True))

print("CHECK before presenting: if these share a created-date cluster and all have")
print("0 stakeholders, they are a bulk import that was never worked -- a data-hygiene")
print("story, not a rep-behaviour story. Different fix, different audience.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(out.win_probability, bins=40, color="tab:blue")
ax[0].set(yscale="log", xlabel="win probability", ylabel="deals (log)",
          title="Open pipeline scores")

band = out.groupby("health", observed=True).agg(
    deals=("display_id", "size"), revenue=("expected_revenue", "sum"))
ax[1].bar(band.index.astype(str), band.revenue / 1e6,
          color=["tab:red", "tab:orange", "tab:green"])
ax[1].set(ylabel="expected revenue ($M)", title="Weighted pipeline by health")
for i, (d, r) in enumerate(zip(band.deals, band.revenue)):
    ax[1].annotate(f"{d} deals", (i, r/1e6), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Limits — read before quoting any number above

**Metrics are optimistic.** Training rows are closed deals in their *final* state;
scoring rows are open deals *mid-flight*. `days_in_sales_cycle` means "final cycle
length" on a closed deal but "age so far" on an open one — same column, two
meanings. Sizing that gap needs backtesting, which needs daily snapshots that
don't exist yet.

**Calibration always extrapolates forward.** 89% of the deals being scored were
created in 2026, while most training labels are 2024–2025 — a resolved cohort can't
be recent, by definition. Narrowing the training window doesn't fix it (measured:
it degrades Σp/actual from 1.36× to 1.54×). This is why Brier and the reliability
curve matter more here than PR-AUC.

**Revenue is softer than the probability.** Amount is real on only ~41% of open
deals. The rest is imputed as a **shrunk geometric mean** by segment × opportunity
type — averaged in log space, then pulled toward the global value by `n/(n+20)` so a
3-deal group doesn't invent its own number. Imputed values span $3.0k (start-up new)
to $52.6k (enterprise new). Quote a range with coverage stated, never a point.

**`segment=2k` is real but unscoreable.** 1 win in 111 closed deals, yet 10% of open
pipeline. Not a data-entry artifact — 302 deals across 139 creation dates since
2025-02-17, $150k median ACV. It's *young*: 185 of 302 still open, so the only
resolved 2k deals are the fast losses. Its 0.90% win rate is censoring, not a rate.

**541 wins is the binding constraint**, not model choice. Deeper models won't help;
more labelled mid-flight examples will.

**Splitting by `opportunity_type` was tried and is worse.** Renewal wins at 28.4% vs
3.8% for new, so two models looks obviously right — but training on new only costs
PR-AUC 0.5781 → 0.5536, 4× the seed spread. `opportunity_type` is already a feature,
so the tree carves that branch itself where it pays, and pooling keeps the 126
renewal/upsell wins feeding every *other* split. Revisit at ~300 renewal wins.

**`has_champion` direction is real, magnitude isn't.** Won deals have one 81.7% of
the time vs 39.3% for lost — partly because champions help, partly because reps
backfill the field on deals they're already closing.

**`poc` means "POC recorded", not "a POC ran."** It's a checkbox with no unset
state (362 True / 12,601 False / 0 null), and 165 deals have POC *notes* written
with the box still False. The True side carries real signal; the False side
conflates "no POC" with "nobody filled it in."

**`segment_unknown` and `source_unknown` scoring as drivers means the model is
partly learning CRM hygiene** — deals with blank fields lose more often. Real
signal, but it's a data-completeness signal, not deal health. Name it as such.

**We cannot yet measure the baseline that decides ship/no-ship** — today's
stage-constant × amount forecast. On closed deals the stage *is* the label
(8 = won, 9 = lost), so the incumbent scores a meaningless 1.0. Needs point-in-time
history.

**This model is single-tenant.** It learns one company's patterns. Scoring a
*different* business needs multiple orgs in training plus tenant-relative features
and a per-tenant calibration layer — see `framework.md` §4c. Ranking probably
transfers; base rates definitely don't.

**Four things unlock from one change** — start snapshotting `dim_opportunity`
daily. That gives point-in-time training rows (fixing the skew above), stage as a
feature, close-date push count, champion-identified date, and honest backtesting.
`fact_opportunity`'s changelog only starts 2026-05-01 while deals go back to
2023-01, so none of it is recoverable retroactively.